**ФИО студента:** [Иванов Иван Иванович]

**Задание:** [выберите между `inpainting` / `outpainting`]

**Группа:** [номер группы]

**Дата сдачи:** [дата]

## Блок настройки окружения

Код для импорта библиотек, для формирования глобальных констант, других конфигураций (если требуется). Постарайтесь здесь зафиксировать все seed, которые сможете

In [ ]:
...

## Блок "Данные"

Код и информация о том, как были подготовлены данные для обучения модели

In [ ]:
...

## Блок "Модель"

Код и информация о том, какая модель была использована

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)


class UNet25D(nn.Module):
    def __init__(self, in_channels=6, out_channels=1, base_channels=32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.encoders = nn.ModuleList([
            ConvBlock(in_channels, base_channels),
            ConvBlock(base_channels, base_channels * 2),
            ConvBlock(base_channels * 2, base_channels * 4),
            ConvBlock(base_channels * 4, base_channels * 8)
        ])
        self.bottleneck = ConvBlock(base_channels * 8, base_channels * 16)
        self.upsamplers = nn.ModuleList([
            nn.ConvTranspose2d(base_channels * 16, base_channels * 8, 2, stride=2),
            nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, stride=2),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2),
            nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        ])
        self.decoders = nn.ModuleList([
            ConvBlock(base_channels * 16, base_channels * 8),
            ConvBlock(base_channels * 8, base_channels * 4),
            ConvBlock(base_channels * 4, base_channels * 2),
            ConvBlock(base_channels * 2, base_channels)
        ])
        self.head = nn.Conv2d(base_channels, out_channels, 1)

    def forward(self, x):
        height, width = x.shape[-2:]
        pad_height = (-height) % 16
        pad_width = (-width) % 16
        x = F.pad(x, (0, pad_width, 0, pad_height))

        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for upsampler, decoder, skip in zip(
            self.upsamplers,
            self.decoders,
            reversed(skips)
        ):
            x = upsampler(x)
            x = torch.cat((x, skip), dim=1)
            x = decoder(x)

        x = self.head(x)
        return x[..., :height, :width]


model = UNet25D()


## Блок "Обучение модели"

Код и информация о том, как был выстроен пайплайн обучения модели (optimizer/scheduler, loss, metrics, training loop, etc). Графики обучения (train + val) по loss + metrics

In [ ]:
...

## Блок "Тестирование модели"

Код и информация о том, как выглядят результаты предсказания на тестовой выборке

In [ ]:
...

## Блок "Инференса модели"

Код и информация о том, как "затягиваются" пропущенные данные в кубе

In [ ]:
...

## Блок сохранения артефактов

Код и информация о том, как сохранялась модель и восстановленный куб (shape финального куба должен совпадать с shape'ом входного куба, который был скачан по ссылке)

In [ ]:
...